# compute_beta_files.ipynb
This file takes .spec files (Peter's format), computes beta and signal-to-noise ratios, and outputs .beta* files.




In [1]:
low_f_range = [2, 5]
high_f_range = [20, 35]

# spec_dir = "example_data/example_spec_files/"
# output_dir = "example_data/example_incomplete_beta_files/"

spec_dir = "/Users/ivandevert/projects/ridgecrest2019_prev/proc/spec/SPECOUT/"
output_dir = "/Users/ivandevert/projects/spectral_falloff_ratio/data/beta_files/"

stn_method = 'time-domain-amplitude'
beta_method = 'test-method-1'


In [2]:
import numpy as np
import scipy
import pandas as pd
import matplotlib.pyplot as plt
import os
import struct
import math
import time

try:
    from geopy import distance
    has_geopy = True
    print("Using geopy package")
except:
    has_geopy = False
    print("Not using geopy package")

Using geopy package


### Define functions to calculate STN ratio and beta

In [3]:
# this is so the functions can have a name attribute
def func_name(r):
    def wrapper(f):
        f.name = r 
        return f 
    return wrapper

@func_name("time-domain-amplitude")
def get_stn(spec):
    """ Compute the signal to noise for a single record:
    1) Take derivative of noise and signal windows separately
    2) Demean combined noise/signal window
    3) STN = mean of abs(signal) / mean of abs(noise)

    Args:
        spec (dict): spec object, returned from read_spec()

    Last Modified:
        2024-06-27
    """
    # take derivative to stabilize the demean
    x1 = np.diff(spec['x1'])    # noise window
    x2 = np.diff(spec['x2'])    # signal window

    # demean entire combined window
    mean = np.mean(np.hstack((x1, x2)))
    x1 = x1 - mean 
    x2 = x2 - mean

    # compute STN ratio
    stn = np.mean(np.abs(x2)) / np.mean(np.abs(x1))

    return stn

@ func_name("test-method-1")
def get_beta(spec, low_f_ind, high_f_ind, df):
    """ Compute the beta parameter:
    1) compute the logmean of a low and high band of the signal spectrum
    2) normalize the mean by the frequency width of each band
    3) beta = high normalized mean / low normalized mean

    Args:
        spec (dict): spec object, returned from read_spec()

    Last Modified:
        2024-06-27
    """
    # assume a linear-scale (non-log) spectrum
    s_low = spec['s2'][low_f_ind[0]:low_f_ind[1]+1]
    s_high = spec['s2'][high_f_ind[0]:high_f_ind[1]+1]

    # log mean value
    low_avg =  np.power(10, np.mean(np.log10(s_low)))  / (df * (low_f_ind[1] - low_f_ind[0]))
    high_avg = np.power(10, np.mean(np.log10(s_high))) / (df * (high_f_ind[1] - high_f_ind[0]))

    beta = high_avg / low_avg
    return beta, low_avg, high_avg

def get_station_name(spec):
    stname = ".".join([el.strip() for el in [spec['stype'], spec['stname'], spec['loccode'], spec['chnm']]])
    return stname

def haversine_distance(origin, destination):
    """
    Calculate the Haversine distance.

    Parameters
    ----------
    origin : tuple of float
        (lat, long)
    destination : tuple of float
        (lat, long)

    Returns
    -------
    distance_in_km : float

    Examples
    --------
    >>> origin = (48.1372, 11.5756)  # Munich
    >>> destination = (52.5186, 13.4083)  # Berlin
    >>> round(distance(origin, destination), 1)
    504.2
    
    Source
    ------
    https://stackoverflow.com/questions/19412462/getting-distance-between-two-points-based-on-latitude-longitude
    """
    lat1, lon1 = origin
    lat2, lon2 = destination
    radius = 6371  # km

    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2) * math.sin(dlat / 2) +
         math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) *
         math.sin(dlon / 2) * math.sin(dlon / 2))
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    d = radius * c

    return d

### Define read/write .betatxt file functions

In [4]:
def write_betatxt_files(df, filedir, low_f_band, high_f_band, beta_method, stn_method):
    """Write information to a .betatxt file. Structure:
    1) Station header. Station name, slat, slon, selev, beta_method, stn_method, low_f_0, low_f_1, high_f_0, high_f_1, nevents
    2) Event lines. Event ID, qmag, beta, low_f_value, high_f_value, stn, deldist

    Sources:

    Last Modified:
        2024-07-18
    """

    for j in range(len(df)):

        stname = df.at[j, 'stname']
        slat = df.at[j, 'slat']
        slon = df.at[j, 'slon']
        selev = df.at[j, 'selev']

        evids = df.at[j, 'evids']
        qmag = df.at[j, 'qmag']
        qlat = df.at[j, 'qlat']
        qlon = df.at[j, 'qlon']
        qdep = df.at[j, 'qlon']
        beta = df.at[j, 'beta']
        stn = df.at[j, 'stn']
        deldist = df.at[j, 'deldist']

        nevents = len(evids)

        filepath = filedir + stname + ".betatxt"
        shead_objs = [
            stname, 
            slat, 
            slon, 
            selev, 
            beta_method, 
            stn_method, 
            low_f_band[0],
            low_f_band[1],
            high_f_band[0],
            high_f_band[1],
            nevents
            ]
        # print(" ".join([str(el) for el in shead_objs]))
        with open(filepath, 'w') as fp:
            fp.write("  ".join([str(el) for el in shead_objs])+"\n")

            for i in range(nevents):
                ln = f"{evids[i]} {qmag[i]:.2f} {qlat[i]:13.8f} {qlon[i]:13.8f} {qdep[i]:11.6f} {beta[i]:9.6e} {stn[i]:7.5e} {deldist[i]:8.5e}\n"
                # print(ln)
                fp.write(ln)
    return

def read_betatxt_files(filedir, ):

    df = pd.DataFrame({
        "stname": pd.arrays.SparseArray(dtype="str", data=[]), 
        "slat": pd.arrays.SparseArray(dtype="float", data=[]), 
        "slon": pd.arrays.SparseArray(dtype="float", data=[]), 
        "selev": pd.arrays.SparseArray(dtype="float", data=[]), 

        "evids": pd.arrays.SparseArray(dtype="object", data=[]), 
        "qmag": pd.arrays.SparseArray(dtype="object", data=[]), 
        "qlat": pd.arrays.SparseArray(dtype="object", data=[]), 
        "qlon": pd.arrays.SparseArray(dtype="object", data=[]), 
        "qdep": pd.arrays.SparseArray(dtype="object", data=[]), 
        "beta": pd.arrays.SparseArray(dtype="object", data=[]), 
        "stn": pd.arrays.SparseArray(dtype="object", data=[]), 
        "deldist": pd.arrays.SparseArray(dtype="object", data=[]), 
    })

    beta_files = [el for el in os.listdir(filedir) if el.endswith('.betatxt')]

    nfiles = len(beta_files)

    for j in range(nfiles):
        filepath = filedir + beta_files[j]

        low_f_band = np.zeros(2, dtype=float)
        high_f_band = np.zeros(2, dtype=float)

        with open(filepath, 'r') as fp:
            line = fp.readline().strip()

        stname, slat, slon, selev, beta_method, stn_method, low_f_band[0], \
        low_f_band[1], high_f_band[0], high_f_band[1], nevents \
        = line.split("  ")

        # print(line)

        column_names = ['evids', 'qmag', 'qlat', 'qlon', 'qdep', 'beta', 'stn', 'deldist']
        data = pd.read_csv(filepath, sep='\\s+', skiprows=1, 
            names=column_names)
        evids = data['evids'].values
        qmag = data['qmag'].values
        qlat = data['qlat'].values
        qlon = data['qlon'].values
        qdep = data['qdep'].values
        beta = data['beta'].values
        stn = data['stn'].values
        deldist = data['deldist'].values
        df.loc[len(df)] = [stname, slat, slon, selev, evids, qmag, qlat, qlon, qdep, beta, stn, deldist]

        # print(data)
    return df


In [5]:
def read_spec(filepath, prec_wf='float32', prec_bp='int32'):
    """Function to read .spec files outputted by Peter Shearer's fortran
    codes. Copied from stressdrop_file_IO.py on 2024-06-25

    Args:
        filepath (str): Path of the .spec file to read

    Returns:
        shead: Spectral method header info
        ehead: Event header info
        spec: Spectra

    Sources:

    Last Modified:
        2024-06-25
    """

    f = open(filepath, 'rb')
    f.seek(0, 2)
    file_size = f.tell()
    f.seek(0, 0)

    spec = []

    shead = {}
    ehead = {}

    junk = struct.unpack('i', f.read(4))[0]
    shead['ispec_method'] = struct.unpack('i', f.read(4))[0]
    shead['ntwind'] = struct.unpack('i', f.read(4))[0]
    shead['nf'] = struct.unpack('i', f.read(4))[0]
    shead['twindoff'] = struct.unpack('f', f.read(4))[0]
    shead['dt'] = struct.unpack('f', f.read(4))[0]
    shead['df'] = struct.unpack('f', f.read(4))[0]    

    junk1 = struct.unpack('i', f.read(4))[0]
    junk2 = struct.unpack('i', f.read(4))[0]
    ehead['efslabel'] = f.read(40).decode('UTF-8')
    ehead['datasource'] = f.read(40).decode('UTF-8')
    ehead['maxnumts'] = struct.unpack('i', f.read(4))[0]
    ehead['numts'] = struct.unpack('i', f.read(4))[0]
    ehead['cuspid'] = struct.unpack('i', f.read(4))[0]
    ehead['qtype'] = f.read(4).decode('UTF-8')
    ehead['qmag1type'] = f.read(4).decode('UTF-8')
    ehead['qmag2type'] = f.read(4).decode('UTF-8')
    ehead['qmag3type'] = f.read(4).decode('UTF-8')
    ehead['qmomenttype'] = f.read(4).decode('UTF-8')
    ehead['qlocqual'] = f.read(4).decode('UTF-8')
    ehead['qfocalqual'] = f.read(4).decode('UTF-8')
    ehead['qlat'] = struct.unpack('f', f.read(4))[0]
    ehead['qlon'] = struct.unpack('f', f.read(4))[0]
    ehead['qdep'] = struct.unpack('f', f.read(4))[0]
    ehead['qsc'] = struct.unpack('f', f.read(4))[0]
    ehead['qmag1'] = struct.unpack('f', f.read(4))[0]
    ehead['qmag2'] = struct.unpack('f', f.read(4))[0]
    ehead['qmag3'] = struct.unpack('f', f.read(4))[0]
    ehead['qmoment'] = struct.unpack('f', f.read(4))[0]
    ehead['qstrike'] = struct.unpack('f', f.read(4))[0]
    ehead['qdip'] = struct.unpack('f', f.read(4))[0]
    ehead['qrake'] = struct.unpack('f', f.read(4))[0]
    ehead['qyr'] = struct.unpack('i', f.read(4))[0]
    ehead['qmon'] = struct.unpack('i', f.read(4))[0]
    ehead['qdy'] = struct.unpack('i', f.read(4))[0]
    ehead['qhr'] = struct.unpack('i', f.read(4))[0]
    ehead['qmn'] = struct.unpack('i', f.read(4))[0]

    # 20 4-byte fields reserved for future uses - skip (80 bytes)
    for idum in range(0, 20):
        dummy = struct.unpack('i', f.read(4))[0]

    # Now loop over all the time series
    for ii in range(0, ehead['numts']):
        # Assemble tshead
        tshead = {}
        junk1 = struct.unpack('i', f.read(4))[0]
        junk2 = struct.unpack('i', f.read(4))[0]
        
        tshead['stname'] = f.read(8).decode('UTF-8')
        tshead['loccode'] = f.read(8).decode('UTF-8')
        tshead['datasource'] = f.read(8).decode('UTF-8')
        tshead['sensor'] = f.read(8).decode('UTF-8')
        tshead['units'] = f.read(8).decode('UTF-8')
        tshead['chnm'] = f.read(4).decode('UTF-8')
        tshead['stype'] = f.read(4).decode('UTF-8')
        tshead['dva'] = f.read(4).decode('UTF-8')
        tshead['pick1q'] = f.read(4).decode('UTF-8')
        tshead['pick2q'] = f.read(4).decode('UTF-8')
        tshead['pick3q'] = f.read(4).decode('UTF-8')
        tshead['pick4q'] = f.read(4).decode('UTF-8')
        tshead['pick1name'] = f.read(4).decode('UTF-8')
        tshead['pick2name'] = f.read(4).decode('UTF-8')
        tshead['pick3name'] = f.read(4).decode('UTF-8')
        tshead['pick4name'] = f.read(4).decode('UTF-8')
        tshead['ppolarity'] = f.read(4).decode('UTF-8')
        tshead['problem'] = f.read(4).decode('UTF-8')
        tshead['npts'] = struct.unpack('i', f.read(4))[0]
        tshead['syr'] = struct.unpack('i', f.read(4))[0]
        tshead['smon'] = struct.unpack('i', f.read(4))[0]
        tshead['sdy'] = struct.unpack('i', f.read(4))[0]
        tshead['shr'] = struct.unpack('i', f.read(4))[0]
        tshead['smn'] = struct.unpack('i', f.read(4))[0]
        tshead['compazi'] = struct.unpack('f', f.read(4))[0]
        tshead['compang'] = struct.unpack('f', f.read(4))[0]
        tshead['gain'] = struct.unpack('f', f.read(4))[0]
        tshead['f1'] = struct.unpack('f', f.read(4))[0]
        tshead['f2'] = struct.unpack('f', f.read(4))[0]
        tshead['dt'] = struct.unpack('f', f.read(4))[0]
        tshead['ssc'] = struct.unpack('f', f.read(4))[0]
        tshead['tdif'] = struct.unpack('f', f.read(4))[0]
        tshead['slat'] = struct.unpack('f', f.read(4))[0]
        tshead['slon'] = struct.unpack('f', f.read(4))[0]
        tshead['selev'] = struct.unpack('f', f.read(4))[0]
        tshead['deldist'] = struct.unpack('f', f.read(4))[0]
        tshead['sazi'] = struct.unpack('f', f.read(4))[0]
        tshead['qazi'] = struct.unpack('f', f.read(4))[0]
        tshead['pick1'] = struct.unpack('f', f.read(4))[0]
        tshead['pick2'] = struct.unpack('f', f.read(4))[0]
        tshead['pick3'] = struct.unpack('f', f.read(4))[0]
        tshead['pick4'] = struct.unpack('f', f.read(4))[0]

        # 20 4-byte fields reserved for future uses - skip
        for idum in range(0, 22):
            dummy = struct.unpack('i', f.read(4))[0]

        # Read the windowed data and spectra
        x1out = np.fromfile(f, dtype = prec_wf, count = shead['ntwind'])
        x2out = np.fromfile(f, dtype = prec_wf, count = shead['ntwind'])

        s1out = np.fromfile(f, dtype = prec_wf, count = shead['nf'])
        s2out = np.fromfile(f, dtype = prec_wf, count = shead['nf'])

        # Bundle tsheader and time-series for this waveform into efsdata, then append to list
        tshead['x1'] = x1out
        tshead['x2'] = x2out
        tshead['s1'] = s1out
        tshead['s2'] = s2out
        spec.append(tshead)

        # some have fewer spectra than numts for some reason
        if file_size - f.tell() < 10: break

    f.close()

    ehead['numts'] = len(spec)

    return shead, ehead, spec

In [54]:
def write_beta(df, output_dir, ):
    """Reciprocal function to readspec. Writes spectrum and associated 
    information to a .spec file.

    Args:
        filepath (str): Path of the .spec file to generate
        shead (_type_): Spectral method header info dict object. Should
            contain ispec_method, ntwind, nf, twindoff, dt, and df.
        spec (_type_): _description_
        prec_wf (str, optional): _description_. Defaults to 'float32'.
        prec_bp (str, optional): _description_. Defaults to 'int32'.

    Sources:

    Last Modified:
        2023-10-30
    """

    prec_wf='float32'
    prec_bp='int32'

    # Change this if the format changes in the future
    FMT_VERSION = int(0)

    for index, row in df.iterrows():
        filepath = output_dir + row['stname'] + ".beta"

        nevents = len(row['evids'])

        with open(filepath, 'wb') as f:
            # Write station header info
            f.write(struct.pack('i', 0))  # Junk
            f.write(struct.pack('i', FMT_VERSION))
            f.write(struct.pack('20s', row['stname'].encode('UTF-8')))
            f.write(struct.pack('f', row['slat']))
            f.write(struct.pack('f', row['slon']))
            f.write(struct.pack('f', row['selev']))
            f.write(struct.pack('30s', row['beta_method'].encode('UTF-8')))
            f.write(struct.pack('30s', row['stn_method'].encode('UTF-8')))
            f.write(struct.pack('f', row['low_f_band'][0]))
            f.write(struct.pack('f', row['low_f_band'][1]))
            f.write(struct.pack('f', row['high_f_band'][0]))
            f.write(struct.pack('f', row['high_f_band'][1]))
            f.write(struct.pack('i', nevents))

            # Write a 4-byte spacer
            f.write(struct.pack('i', 0))


            # Write event data
            for i in range(nevents):

                f.write(struct.pack('ii', 3, 1))  # Junk
                f.write(struct.pack('20s', str(row['evids'][i]).encode('UTF-8')))
                f.write(struct.pack('f', row['qmag'][i]))
                f.write(struct.pack('f', row['qlat'][i]))
                f.write(struct.pack('f', row['qlon'][i]))
                f.write(struct.pack('f', row['qdep'][i]))
                f.write(struct.pack('f', row['beta'][i]))
                f.write(struct.pack('f', row['stn'][i]))
                f.write(struct.pack('f', row['deldist'][i]))

                # Write a 4-byte spacer
                f.write(struct.pack('i', 0))

In [70]:
def read_beta(beta_dir):
    
    beta_files = [el for el in os.listdir(beta_dir) if el.endswith('.beta')]
    beta_files.sort()
    nfiles = len(beta_files)

    df = pd.DataFrame({
        "stname": pd.arrays.SparseArray(dtype="str", data=[]), 
        "slat": pd.arrays.SparseArray(dtype="float", data=[]), 
        "slon": pd.arrays.SparseArray(dtype="float", data=[]), 
        "selev": pd.arrays.SparseArray(dtype="float", data=[]), 

        "event_id": pd.arrays.SparseArray(dtype="object", data=[]), 
        "qmag": pd.arrays.SparseArray(dtype="object", data=[]), 
        "qlat": pd.arrays.SparseArray(dtype="object", data=[]), 
        "qlon": pd.arrays.SparseArray(dtype="object", data=[]), 
        "qdep": pd.arrays.SparseArray(dtype="object", data=[]), 
        "beta": pd.arrays.SparseArray(dtype="object", data=[]), 
        "stn": pd.arrays.SparseArray(dtype="object", data=[]), 
        "deldist": pd.arrays.SparseArray(dtype="object", data=[]), 
    })

    for i in range(nfiles):
        filepath = beta_dir + beta_files[i]

        f = open(filepath, 'rb')
        f.seek(0, 2)
        file_size = f.tell()
        f.seek(0, 0)

        junk = struct.unpack('i', f.read(4))[0]
        FMT_VERSION = struct.unpack('i', f.read(4))[0]

        if FMT_VERSION == 0:
            stname = f.read(20).decode('UTF-8')
            slat = struct.unpack('f', f.read(4))[0]
            slon = struct.unpack('f', f.read(4))[0]
            selev = struct.unpack('f', f.read(4))[0]
            beta_method = f.read(30).decode('UTF-8')
            stn_method = f.read(30).decode('UTF-8')
            low_f_band = (struct.unpack('f', f.read(4))[0], 
                          struct.unpack('f', f.read(4))[0])
            high_f_band = (struct.unpack('f', f.read(4))[0],
                           struct.unpack('f', f.read(4))[0])
            nevents = struct.unpack('i', f.read(4))[0]

            evids = np.zeros(nevents, dtype='<U20')
            qmag = np.zeros(nevents, dtype=float)
            qlat = np.zeros_like(qmag)
            qlon = np.zeros_like(qmag)
            qdep = np.zeros_like(qmag)
            beta = np.zeros_like(qmag)
            stn = np.zeros_like(qmag)
            deldist = np.zeros_like(qmag)
            for j in range(nevents):
                junk1 = struct.unpack('i', f.read(4))[0]
                junk2 = struct.unpack('i', f.read(4))[0]
                # print(junk1, junk2)
                evids[j] = f.read(20).decode('UTF-8')
                qmag[j] = struct.unpack('f', f.read(4))[0]
                qlat[j] = struct.unpack('f', f.read(4))[0]
                qlon[j] = struct.unpack('f', f.read(4))[0]
                qdep[j] = struct.unpack('f', f.read(4))[0]
                beta[j] = struct.unpack('f', f.read(4))[0]
                stn[j] = struct.unpack('f', f.read(4))[0]
                deldist[j] = struct.unpack('f', f.read(4))[0]
                junk = struct.unpack('i', f.read(4))[0]
            df.loc[len(df)] = [stname, slat, slon, selev, evids, qmag, qlat, qlon, qdep, beta, stn, deldist]
    return df


In [6]:

# Get a list of all .spec files in spec_dir
spec_event_ids = [int(el.split('.')[0]) for el in os.listdir(spec_dir) \
                    if el.endswith(".spec")]
spec_filepaths = [spec_dir + el for el in os.listdir(spec_dir) \
                    if el.endswith(".spec")]

nevents = len(spec_event_ids)

print(f"{nevents} .spec files found")

12943 .spec files found


In [7]:
assert get_stn.name == stn_method, "Check get_stn function names/methods"
assert get_beta.name == beta_method, "Check get_beta function names/methods"

# open a .spec file as a test
shead, ehead, spec = read_spec(spec_filepaths[0])

deldist_avg = 0
for sp in spec:
    deldist_avg += sp['deldist']
deldist_avg /= len(spec)
if deldist_avg <= 10:
    dist_km = False
    print(f"Average deldist is {deldist_avg:.4f}. Interpreting this as degrees.")
elif deldist_avg > 10:
    dist_km = True
    print(f"Average deldist is {deldist_avg:.4f}. Interpreting this as km.")


f = np.arange(shead['nf']) * shead['df']


low_f_ind = np.array([np.argmin(np.abs(f - el)) for el in low_f_range])
high_f_ind = np.array([np.argmin(np.abs(f - el)) for el in high_f_range])

low_f_band = f[low_f_ind]
high_f_band = f[high_f_ind]

Average deldist is 0.5138. Interpreting this as degrees.


In [8]:
# nevents = 100
# dfs = [[]] * nevents
# for i in range(nevents):
#     shead, ehead, spec = read_spec(spec_filepaths[i])

#     nspec = len(spec)
#     A = [[]]*nspec
#     for j in range(nspec):
#         A[j] = {**shead, **ehead, **spec[j]}
#     dfs[i] = pd.DataFrame(A)
# df = pd.concat(dfs)



In [9]:
import os 
import importlib 
import stressdrop_file_IO as sdio

importlib.reload(sdio)

import math

def convert_size(size_bytes):
   if size_bytes == 0:
       return "0B"
   size_name = ("B", "KB", "MB", "GB", "TB", "PB", "EB", "ZB", "YB")
   i = int(math.floor(math.log(size_bytes, 1024)))
   p = math.pow(1024, i)
   s = round(size_bytes / p, 2)
   return "%s %s" % (s, size_name[i])

# df = sdio.write_spec_pkl(spec_dir, output_dir)

### Compute and write .betatxt files
Each station has one .betatxt file with all events recorded by that station, while each .spec file represents one event with all stations recording that event. The .betatxt file is essentially a reorganization of the same data, but with a few added fields (beta and stn, for now).

New way, using explode and groupby

In [11]:
# Each station/row will have a single value for each of (stname, slat, slon,
# selev), while the fields (evids, qmag, beta, stn, deldist) will be variable
# but equal length arrays containing one entry per event.
columns = {
    "stname"  : "object",
    "slat"    : "float", 
    "slon"    : "float", 
    "selev"   : "float", 
    "evids"   : "object", 
    "qmag"    : "object", 
    "qlat"    : "object", 
    "qlon"    : "object", 
    "qdep"    : "object", 
    "beta"    : "object", 
    "stn"     : "object", 
    "deldist" : "object", 
}

# initialize the pandas DataFrame with the above columns and datatypes
df = pd.DataFrame({key: np.empty(dtype=val, shape=0) for key, val in columns.items()})

t0 = time.time()
ii = 0
for i in range(nevents):
    try:
        shead, ehead, spec = read_spec(spec_filepaths[i])

        qlat = ehead['qlat']
        qlon = ehead['qlon']
        qdep = ehead['qdep']
        qmag = ehead['qmag1']

        evid = ehead['cuspid']


        nspec = ehead['numts']

        for j in range(nspec):
            sp = spec[j]
            stname = get_station_name(sp)

            # this is pretty slow
            dist = distance.distance((qlat, qlon), (sp['slat'], sp['slon'])).km

            # beta
            beta, low_avg, high_avg = get_beta(sp, low_f_ind, high_f_ind, shead["df"])

            # signal to noise
            stn = get_stn(sp)

            # if a row for the station exists, append event information to
            # select columns
            if stname in df['stname'].values:
                ind = np.where(df['stname'].values==stname)[0][0]
                df.at[ind, 'evids'] = np.append(df.at[ind, 'evids'], evid)
                df.at[ind, 'qmag'] = np.append(df.at[ind, 'qmag'], qmag)
                df.at[ind, 'qlat'] = np.append(df.at[ind, 'qlat'], qlat)
                df.at[ind, 'qlon'] = np.append(df.at[ind, 'qlon'], qlon)
                df.at[ind, 'qdep'] = np.append(df.at[ind, 'qdep'], qdep)
                df.at[ind, 'beta'] = np.append(df.at[ind, 'beta'], beta)
                df.at[ind, 'stn'] = np.append(df.at[ind, 'stn'], stn)
                df.at[ind, 'deldist'] = np.append(df.at[ind, 'deldist'], dist)


            else:
                # if the station doesn't have a row, append a new row for 
                # the station
                evids = np.array([evid, ])
                qmags = np.array([qmag, ])
                qlats = np.array([qlat, ])
                qlons = np.array([qlon, ])
                qdeps = np.array([qdep, ])
                beta = np.array([beta, ])
                stn = np.array([stn, ])
                deldist = np.array([dist, ])
                df.loc[len(df)] = [stname, sp['slat'], sp['slon'], sp['selev'], evids, qmags, qlats, qlons, qdeps, beta, stn, deldist]
        ii += 1
    except:
        print(f"Error with {spec_filepaths[i]}")

df['beta_method'] = beta_method
df['stn_method'] = stn_method
df['low_f_band'] = [list(low_f_band.astype(float))] * len(df)
df['high_f_band'] = [list(low_f_band.astype(float))] * len(df)
t1 = time.time()

print(f"{t1-t0:.3} s reading and computing from .spec files")
print(f"{(t1-t0)*1000/ii:.2f}ms per .spec file")

print("Writing to .betatxt files...", end="")
t0 = time.time()
write_beta(df, output_dir)
# write_betatxt_files(df, output_dir, low_f_band, high_f_band, beta_method, stn_method)
t1 = time.time()
print("Done.\n")
print(f"{t1-t0:.3} s writing .betatxt files")
print(f"{(t1-t0)*1000/nevents:.2f}ms per .spec file")

/var/folders/n9/y3b20y1x2dx7n3qq1qn2cn2m000jzz/T/ipykernel_19440/3693232668.py:31: RuntimeWarning: invalid value encountered in scalar divide
  stn = np.mean(np.abs(x2)) / np.mean(np.abs(x1))
/var/folders/n9/y3b20y1x2dx7n3qq1qn2cn2m000jzz/T/ipykernel_19440/3693232668.py:31: RuntimeWarning: invalid value encountered in scalar divide
  stn = np.mean(np.abs(x2)) / np.mean(np.abs(x1))


Error with /Users/ivandevert/projects/ridgecrest2019_prev/proc/spec/SPECOUT/38533903.spec


/var/folders/n9/y3b20y1x2dx7n3qq1qn2cn2m000jzz/T/ipykernel_19440/3693232668.py:31: RuntimeWarning: invalid value encountered in scalar divide
  stn = np.mean(np.abs(x2)) / np.mean(np.abs(x1))
/var/folders/n9/y3b20y1x2dx7n3qq1qn2cn2m000jzz/T/ipykernel_19440/3693232668.py:31: RuntimeWarning: invalid value encountered in scalar divide
  stn = np.mean(np.abs(x2)) / np.mean(np.abs(x1))


2.96e+03 s reading and computing from .spec files
228.46ms per .spec file
Writing to .betatxt files...Done.

20.4 s writing .betatxt files
1.58ms per .spec file


In [13]:
# # test speed of geopy distance function vs haversine distance function.
# # geopy distance should be more accurate

# from geopy import distance

# N = 1000

# lats1 = np.random.rand(N) * 180 - 90
# lons1 = np.random.rand(N) * 360 - 180

# lats2 = np.random.rand(N) * 180 - 90
# lons2 = np.random.rand(N) * 360 - 180

# distances = np.zeros(N, dtype=float)

# t0 = time.time()
# for i in range(N):
#     distances[i] = distance.distance((lats1[i], lons1[i]), (lats2[i], lons2[i])).km
# t1 = time.time()
# print(f"{t1-t0:.3} s calculating distances for {N} sets of points")
# print(f"{(t1-t0)*1E6/N:.2f}μs per set of points")

In [14]:
# ### get_stn()
# t0 = time.time()

# for j in range(len(spec)):
#     a = get_stn(spec[j])

# t1 = time.time()

# print("get_stn():")
# print(f"{t1-t0:.4f} s for {len(spec)} iterations")
# print(f"{(t1-t0)*1000/len(spec):.4f} ms per iteration")

# ### get_beta()
# t0 = time.time()

# for j in range(len(spec)):
#     a = get_beta(spec[j], low_f_ind, high_f_ind, shead["df"])

# t1 = time.time()

# print("get_beta():")
# print(f"{t1-t0:.4f} s for {len(spec)} iterations")
# print(f"{(t1-t0)*1000/len(spec):.4f} ms per iteration")